In [2]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# -----------------------------
# Part A: Image Representation
# -----------------------------

def extract_local_zip(zip_path="/content/archive.zip", data_dir="att_faces"):
    """Extract the local zip file containing the ORL faces dataset."""
    if not os.path.exists(data_dir):
        print(f"Extracting local dataset from {zip_path} into '{data_dir}'...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
        print("Dataset extracted successfully.")
    else:
        print(f"'{data_dir}' directory already exists.")

def load_orl_dataset(data_dir):
    """Load ORL faces, return flattened image matrix (n_samples, n_pixels) and labels."""
    images, labels = [], []
    for root, dirs, files in os.walk(data_dir):
        for file in sorted(files):
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.pgm')):
                img_path = os.path.join(root, file)
                img = Image.open(img_path).convert('L')  # Grayscale
                arr = np.array(img, dtype=np.float64)
                images.append(arr.flatten())

                # Subject name is usually the parent folder name (e.g. 's1', 's2')
                subject_dir = os.path.basename(root)
                label = int(''.join(filter(str.isdigit, subject_dir)))
                labels.append(label)
    X = np.vstack(images)
    y = np.array(labels)
    return X, y

def make_split(X, y, test_size=0.25, random_state=42):
    """Stratified split so every subject appears in train and test."""
    return train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )

# -----------------------------
# Part B: PCA / Eigenfaces (NumPy only)
# -----------------------------

def pca_numpy(X, n_components=None):
    """
    PCA via covariance eigendecomposition and SVD.
    Returns:
      mean: (n_features,)
      eigenvectors: (n_features, n_components)
      eigenvalues: (n_components,)
      explained_variance_ratio: (n_components,)
    """
    n_samples, n_features = X.shape
    mean = X.mean(axis=0)
    X_centered = X - mean

    # Covariance eigendecomposition
    cov = np.cov(X_centered, rowvar=False)
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    if n_components is not None:
        eigenvectors = eigenvectors[:, :n_components]
        eigenvalues = eigenvalues[:n_components]

    total_var = eigenvalues.sum()
    explained_variance_ratio = eigenvalues / total_var

    # SVD for comparison
    U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
    svd_eigenvectors = Vt.T
    svd_eigenvalues = (S ** 2) / (n_samples - 1)

    return mean, eigenvectors, eigenvalues, explained_variance_ratio, svd_eigenvectors, svd_eigenvalues

def choose_n_components(explained_variance_ratio, threshold=0.90):
    """Minimum components to reach threshold cumulative variance."""
    cumsum = np.cumsum(explained_variance_ratio)
    return int(np.argmax(cumsum >= threshold) + 1)

def project(X, mean, eigenvectors):
    return (X - mean) @ eigenvectors

def reconstruct(Z, mean, eigenvectors):
    return Z @ eigenvectors.T + mean

def plot_explained_variance(explained_variance_ratio, n_components, save_path="explained_variance.png"):
    cumsum = np.cumsum(explained_variance_ratio)
    plt.figure(figsize=(8, 5))
    plt.plot(range(1, len(explained_variance_ratio) + 1), cumsum, 'b-')
    plt.axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
    plt.axvline(x=n_components, color='g', linestyle='--', label=f'n={n_components}')
    plt.xlabel('Number of Components')
    plt.ylabel('Cumulative Explained Variance')
    plt.title('Cumulative Explained Variance')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def plot_eigenfaces(eigenvectors, mean, img_shape, n_show=12, save_path="eigenfaces.png"):
    fig, axes = plt.subplots(3, 4, figsize=(10, 8))
    for i, ax in enumerate(axes.flat):
        if i < n_show:
            face = eigenvectors[:, i].reshape(img_shape) + mean.reshape(img_shape)
            ax.imshow(face, cmap='gray')
            ax.set_title(f'Eigenface {i+1}')
        ax.axis('off')
    plt.suptitle('Top Eigenfaces')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def plot_reconstructions(X_test, y_test, mean, eigenvectors, img_shape, n_show=8, save_path="reconstructions.png"):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for i, ax in enumerate(axes.flat):
        idx = i
        original = X_test[idx].reshape(img_shape)
        Z = project(X_test[idx:idx+1], mean, eigenvectors)
        rec = reconstruct(Z, mean, eigenvectors)[0].reshape(img_shape)
        ax.imshow(np.hstack([original, rec]), cmap='gray')
        ax.set_title(f'Orig | Rec (ID {y_test[idx]})')
        ax.axis('off')
    plt.suptitle('Reconstruction Examples')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

# -----------------------------
# Part C: Recognition in PCA Space
# -----------------------------

def recognize_by_similarity(Z_train, y_train, Z_test):
    """Nearest neighbor in PCA space using Euclidean distance."""
    predictions = []
    for z in Z_test:
        dists = np.linalg.norm(Z_train - z, axis=1)
        nearest = np.argmin(dists)
        predictions.append(y_train[nearest])
    return np.array(predictions)

def compare_raw_vs_pca(X_train, y_train, X_test, Z_train, Z_test):
    """Compare raw-pixel NN vs PCA-space NN."""
    # Raw pixel NN
    raw_preds = []
    for x in X_test:
        dists = np.linalg.norm(X_train - x, axis=1)
        raw_preds.append(y_train[np.argmin(dists)])
    raw_preds = np.array(raw_preds)

    # PCA NN
    pca_preds = recognize_by_similarity(Z_train, y_train, Z_test)

    raw_acc = accuracy_score(y_test, raw_preds)
    pca_acc = accuracy_score(y_test, pca_preds)
    return raw_acc, pca_acc

# -----------------------------
# Part D: Traditional ML Classifiers
# -----------------------------

def train_classifiers(Z_train, y_train, Z_test):
    results = {}

    knn = KNeighborsClassifier(n_neighbors=3)
    knn.fit(Z_train, y_train)
    results['KNN (k=3)'] = knn.score(Z_test, y_test)

    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(Z_train, y_train)
    results['Logistic Regression'] = lr.score(Z_test, y_test)

    svm = SVC(kernel='rbf', C=1.0, gamma='scale')
    svm.fit(Z_train, y_train)
    results['SVM (RBF)'] = svm.score(Z_test, y_test)

    return results

# -----------------------------
# Main
# -----------------------------

def main():
    zip_path = "/content/archive.zip"
    data_dir = "att_faces"

    # Extract local archive file
    extract_local_zip(zip_path, data_dir)

    X, y = load_orl_dataset(data_dir)
    print(f"Loaded {X.shape[0]} images, {len(np.unique(y))} subjects")
    print(f"Image shape per sample: {X.shape[1]} pixels")

    X_train, X_test, y_train, y_test = make_split(X, y)

    # Automatically determine image aspect ratio (standard ORL is usually 92x112 or 112x92)
    # We'll default to (112, 92) and reshape gracefully based on features
    if X.shape[1] == 10304:
        img_shape = (112, 92)
    else:
        side = int(np.sqrt(X.shape[1]))
        img_shape = (side, side)

    # PCA
    mean, eigenvectors, eigenvalues, evr, svd_vecs, svd_vals = pca_numpy(X_train)
    n_components = choose_n_components(evr, threshold=0.90)
    print(f"Components for 90% variance: {n_components}")

    # Compare covariance vs SVD
    diff_vecs = np.linalg.norm(eigenvectors[:, :n_components] - svd_vecs[:, :n_components])
    print(f"Max eigenvector difference (cov vs SVD): {diff_vecs:.6f}")

    # Keep only needed components
    eigenvectors = eigenvectors[:, :n_components]

    # Plots
    plot_explained_variance(evr, n_components)
    plot_eigenfaces(eigenvectors, mean, img_shape)

    # Projection
    Z_train = project(X_train, mean, eigenvectors)
    Z_test = project(X_test, mean, eigenvectors)

    # Reconstruction plot
    plot_reconstructions(X_test, y_test, mean, eigenvectors, img_shape)

    # Storage comparison
    original_storage = X_train.shape[1] * X_train.shape[0]
    reduced_storage = Z_train.shape[1] * Z_train.shape[0]
    print(f"Original storage: {original_storage} floats")
    print(f"Reduced storage: {reduced_storage} floats")
    print(f"Compression ratio: {original_storage / reduced_storage:.2f}x")

    # Recognition
    pca_preds = recognize_by_similarity(Z_train, y_train, Z_test)
    pca_acc = accuracy_score(y_test, pca_preds)
    print(f"PCA similarity recognition accuracy: {pca_acc:.4f}")

    raw_acc, pca_acc2 = compare_raw_vs_pca(X_train, y_train, X_test, Z_train, Z_test)
    print(f"Raw pixel NN accuracy: {raw_acc:.4f}")
    print(f"PCA NN accuracy (recomputed): {pca_acc2:.4f}")

    # ML classifiers
    ml_results = train_classifiers(Z_train, y_train, Z_test)
    print("\nML Classifier Accuracies:")
    for name, acc in ml_results.items():
        print(f"  {name}: {acc:.4f}")

    # Final comparison table
    print("\nFinal Comparison:")
    print(f"  Raw Pixel NN: {raw_acc:.4f}")
    print(f"  PCA NN: {pca_acc:.4f}")
    for name, acc in ml_results.items():
        print(f"  {name}: {acc:.4f}")

if __name__ == "__main__":
    main()

Extracting local dataset from /content/archive.zip into 'att_faces'...
Dataset extracted successfully.
Loaded 400 images, 40 subjects
Image shape per sample: 10304 pixels
Components for 90% variance: 95
Max eigenvector difference (cov vs SVD): 14.422205
Original storage: 3091200 floats
Reduced storage: 28500 floats
Compression ratio: 108.46x
PCA similarity recognition accuracy: 0.9700
Raw pixel NN accuracy: 0.9600
PCA NN accuracy (recomputed): 0.9700

ML Classifier Accuracies:
  KNN (k=3): 0.9700
  Logistic Regression: 0.9800
  SVM (RBF): 0.9800

Final Comparison:
  Raw Pixel NN: 0.9600
  PCA NN: 0.9700
  KNN (k=3): 0.9700
  Logistic Regression: 0.9800
  SVM (RBF): 0.9800
